# Code - Modality 1 (DeepFashion images)

This notebook keeps the data, noise process, U-Net, training loop, sampling, and FID steps explicit. It uses the same 64×64 class-conditional architecture and cosine diffusion schedule as `fashion_cfg_diffusion.py`.


In [ ]:
# Imports and shared settings
from pathlib import Path
import math
import random
import time
from collections import defaultdict

import numpy as np
from PIL import Image, ImageOps
import torch
from torch.utils.data import DataLoader, Dataset, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42
image_size = 64
batch_size = 16
num_classes = 50  # The coarse DeepFashion category table has 50 names.
null_class = num_classes  # Extra embedding used when the label is dropped for CFG.
data_root = Path("/home/pranshav/Documents/PycharmProjects/fashion")
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)


In [ ]:
# Print device to confirm:
print(f"Using device: {device}")

In [ ]:
# Load captions.csv using the same relative image paths as FashionDataset.
import csv

captions_path = data_root / "captions.csv"
with captions_path.open(encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    if reader.fieldnames != ["image_path", "caption"]:
        raise ValueError(f"Unexpected caption columns: {reader.fieldnames}")
    captions_by_image = {row["image_path"]: row["caption"] for row in reader}

print(f"Loaded {len(captions_by_image):,} captions")


In [ ]:
# Load DeepFashion's official train/validation partitions and category labels.
def read_rows(path):
    with Path(path).open(encoding="utf-8") as file:
        next(file)  # Number of records.
        next(file)  # Column names.
        for line in file:
            if line.strip():
                yield line.split()


class FashionDataset(Dataset):
    def __init__(self, root, split, captions, image_size=64, flip=False, limit=0, seed=42):
        self.root = Path(root)
        self.captions = captions
        self.image_size = image_size
        self.flip = flip
        annotation = self.root / "Anno_coarse"

        self.classes = [" ".join(row[:-1]) for row in read_rows(annotation / "list_category_cloth.txt")]
        labels = {row[0]: int(row[1]) - 1 for row in read_rows(annotation / "list_category_img.txt")}
        boxes = {row[0]: tuple(map(int, row[1:5])) for row in read_rows(annotation / "list_bbox.txt")}
        self.records = [(path, labels[path], boxes[path])
                        for path, partition in read_rows(self.root / "Eval/list_eval_partition.txt")
                        if partition == split and path in labels and path in boxes]
        if not self.records:
            raise ValueError(f"No {split} images found under {self.root}")

        if limit and limit < len(self.records):
            # The small validation set stays balanced across the available classes.
            groups = defaultdict(list)
            for record in self.records:
                groups[record[1]].append(record)
            rng = random.Random(seed)
            for group in groups.values():
                rng.shuffle(group)
            selected = []
            class_ids = sorted(groups)
            while len(selected) < limit and class_ids:
                for class_id in class_ids[:]:
                    if len(selected) == limit:
                        break
                    if groups[class_id]:
                        selected.append(groups[class_id].pop())
                    else:
                        class_ids.remove(class_id)
            self.records = selected

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        relative_path, label, (x1, y1, x2, y2) = self.records[index]
        with Image.open(self.root / relative_path) as source:
            image = source.convert("RGB")
            x1, x2 = max(0, x1), min(image.width, x2)
            y1, y2 = max(0, y1), min(image.height, y2)
            if x2 > x1 and y2 > y1:
                image = image.crop((x1, y1, x2, y2))
            image = ImageOps.pad(image, (self.image_size, self.image_size),
                                 method=Image.Resampling.BICUBIC, color=(255, 255, 255))
            if self.flip and random.random() < 0.5:
                image = ImageOps.mirror(image)
            pixels = np.asarray(image, dtype=np.uint8).copy()
        # Convert RGB pixels from [0, 255] to [-1, 1].
        x0 = torch.from_numpy(pixels).permute(2, 0, 1).float().div_(127.5).sub_(1)
        return x0, label, self.captions[relative_path]


train_data = FashionDataset(data_root, "train", captions_by_image, image_size, flip=True)
val_data = FashionDataset(data_root, "val", captions_by_image, image_size, limit=2048, seed=seed)
active_class_ids = sorted({label for _, label, _ in train_data.records})
class_names = train_data.classes
assert len(class_names) == num_classes

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=(device.type == "cuda"))
print(f"Train: {len(train_data):,}; validation: {len(val_data):,}; "
      f"active classes: {len(active_class_ids)} of {num_classes}")

# A dataset item now contains its image, category, and caption together.
image, class_id, caption = train_data[0]
print(f"Example {class_names[class_id]} caption: {caption}")


In [ ]:
# Define the same 1,000-step cosine noise schedule as fashion_cfg_diffusion.py.
K = 1000
step_fraction = torch.linspace(0, K, K + 1, device=device, dtype=torch.float64) / K
cosine_curve = torch.cos((step_fraction + 0.008) / 1.008 * math.pi / 2).square()
cosine_curve = cosine_curve / cosine_curve[0]
betas = (1 - cosine_curve[1:] / cosine_curve[:-1]).clamp(0.0001, 0.9999).float()
alpha_bars = torch.cumprod(1 - betas, dim=0)  # Indices 0..999; no clean-image index.


def Add_Noise(x0, k):
    '''Draw Gaussian noise and construct x_k for one random timestep per image.'''
    epsilon = torch.randn_like(x0)
    alpha = alpha_bars[k][:, None, None, None]
    xk = alpha.sqrt() * x0 + (1 - alpha).sqrt() * epsilon
    return xk, epsilon


In [ ]:
# Define the same conditional U-Net as fashion_cfg_diffusion.py, explicitly.
from torch import nn
import torch.nn.functional as F

base_channels = 64


def timestep_embedding(k, width):
    '''Sinusoidal timestep features, before a learned projection.'''
    half = width // 2
    frequencies = torch.exp(-math.log(10000) * torch.arange(half, device=k.device)
                            / max(half - 1, 1))
    angles = k.float()[:, None] * frequencies[None]
    result = torch.cat([angles.sin(), angles.cos()], dim=-1)
    return F.pad(result, (0, width - result.shape[-1]))


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, embedding_width):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.embedding = nn.Linear(embedding_width, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip = (nn.Conv2d(in_channels, out_channels, 1)
                     if in_channels != out_channels else nn.Identity())

    def forward(self, x, embedding):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.embedding(F.silu(embedding))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Attention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        q, k, v = self.qkv(self.norm(x)).chunk(3, dim=1)
        q = q.flatten(2).transpose(1, 2)
        k = k.flatten(2).transpose(1, 2)
        v = v.flatten(2).transpose(1, 2)
        h = F.scaled_dot_product_attention(q, k, v)
        h = h.transpose(1, 2).reshape(batch, channels, height, width)
        return x + self.out(h)


class DiffusionUNet(nn.Module):
    def __init__(self, classes=num_classes, base=base_channels):
        super().__init__()
        self.base = base
        width = base * 4
        self.time_mlp = nn.Sequential(nn.Linear(base, width), nn.SiLU(), nn.Linear(width, width))
        self.class_embedding = nn.Embedding(classes + 1, width)
        self.input = nn.Conv2d(3, base, 3, padding=1)
        self.down1 = ResBlock(base, base, width)
        self.stride1 = nn.Conv2d(base, base, 3, stride=2, padding=1)
        self.down2 = ResBlock(base, base * 2, width)
        self.stride2 = nn.Conv2d(base * 2, base * 2, 3, stride=2, padding=1)
        self.middle1 = ResBlock(base * 2, base * 2, width)
        self.attention = Attention(base * 2)  # Bottleneck self-attention at 16×16.
        self.middle2 = ResBlock(base * 2, base * 2, width)
        self.up2 = ResBlock(base * 4, base * 2, width)
        self.up1 = ResBlock(base * 3, base, width)
        self.output_norm = nn.GroupNorm(8, base)
        self.output = nn.Conv2d(base, 3, 3, padding=1)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, xk, k, class_labels):
        embedding = (self.time_mlp(timestep_embedding(k, self.base))
                     + self.class_embedding(class_labels))
        x = self.input(xk)
        skip1 = self.down1(x, embedding)              # [B, base, 64, 64]
        skip2 = self.down2(self.stride1(skip1), embedding)  # [B, 2*base, 32, 32]
        x = self.middle2(self.attention(self.middle1(self.stride2(skip2), embedding)), embedding)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.up2(torch.cat([x, skip2], dim=1), embedding)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.up1(torch.cat([x, skip1], dim=1), embedding)
        return self.output(F.silu(self.output_norm(x)))  # Predict RGB noise.


model = DiffusionUNet().to(device)


In [ ]:
# Train the conditional noise predictor, with 10% class-label dropout for CFG.
lr = 2e-4
drop_class_percentage = 0.1
num_epochs = 50
patience = 10
min_delta = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
loss_fn = nn.MSELoss()
best_val = significant_best = float("inf")
stale_epochs = 0
best_weights = None


@torch.no_grad()
def Validation_MSE(model):
    model.eval()
    # Reuse the same corruption draws at each epoch for a fair comparison.
    rng = torch.Generator(device=device).manual_seed(seed + 1000)
    total_squared_error = num_pixels = 0
    for x0, labels, captions in val_loader:
        x0, labels = x0.to(device), labels.to(device)
        epsilon = torch.randn(x0.shape, device=device, generator=rng)
        k = torch.randint(K, (len(x0),), device=device, generator=rng)
        alpha = alpha_bars[k][:, None, None, None]
        xk = alpha.sqrt() * x0 + (1 - alpha).sqrt() * epsilon
        predicted = model(xk, k, labels)
        total_squared_error += F.mse_loss(predicted.float(), epsilon, reduction="sum").item()
        num_pixels += epsilon.numel()
    return total_squared_error / num_pixels


for epoch in range(num_epochs):
    model.train()
    total_loss = num_images = 0
    for x0, labels, captions in train_loader:
        x0, labels = x0.to(device), labels.to(device)
        k = torch.randint(K, (len(x0),), device=device)  # One timestep 0..999 per image.
        xk, epsilon = Add_Noise(x0, k)
        dropped = torch.rand(len(x0), device=device) < drop_class_percentage
        model_labels = labels.masked_fill(dropped, null_class)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            predicted_noise = model(xk, k, model_labels)
            loss = loss_fn(predicted_noise.float(), epsilon.float())
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(x0)
        num_images += len(x0)

    val_mse = Validation_MSE(model)
    if val_mse < best_val:
        best_val = val_mse
        # Keep CPU copies so later updates do not change the saved best model.
        best_weights = {name: tensor.detach().cpu().clone()
                        for name, tensor in model.state_dict().items()}
    if val_mse < significant_best - min_delta:
        significant_best, stale_epochs = val_mse, 0
    else:
        stale_epochs += 1
    print(f"Epoch {epoch + 1}: train MSE={total_loss / num_images:.5f}, "
          f"validation MSE={val_mse:.5f}, best={best_val:.5f}")
    if patience and stale_epochs >= patience:
        print(f"Early stopping after {epoch + 1} epochs")
        break

if best_weights is not None:
    model.load_state_dict(best_weights)  # Use the validation-selected model below.


In [ ]:
# Optional: save the trained (validation-selected) model.
torch.save(model.state_dict(), "fashion_diffusion_best.pt")

# Or load a checkpoint produced by fashion_cfg_diffusion.py. Rebuild the model
# with the checkpoint's base width (the existing fashion-cfg-run uses 32):
# checkpoint = torch.load(data_root / "fashion-cfg-run" / "best.pt",
#                         map_location=device, weights_only=False)
# model = DiffusionUNet(base=checkpoint["config"]["base"]).to(device)
# model.load_state_dict(checkpoint["model"])
# model.eval()


In [ ]:
# Reverse diffusion: deterministic DDIM-style steps with classifier-free guidance.
@torch.no_grad()
def Sample_Images(model, class_labels, guidance_scale=3.0, generator=None, sample_steps=50):
    model.eval()
    class_labels = torch.as_tensor(class_labels, device=device, dtype=torch.long)
    if not 1 <= sample_steps <= K:
        raise ValueError("sample_steps must be between 1 and K")
    if any(label.item() not in active_class_ids for label in class_labels):
        raise ValueError("A requested class has no training images")
    batch = len(class_labels)
    null_labels = torch.full_like(class_labels, null_class)
    xk = torch.randn(batch, 3, image_size, image_size, device=device, generator=generator)
    steps = torch.linspace(K - 1, 0, sample_steps, device=device).round().long().unique_consecutive()

    for index, step in enumerate(steps):
        k = step.expand(batch)
        if guidance_scale == 0:
            epsilon = model(xk, k, null_labels)
        elif guidance_scale == 1:
            epsilon = model(xk, k, class_labels)
        else:
            # Batch conditional and unconditional predictions in one forward pass.
            pair = model(torch.cat([xk, xk]), torch.cat([k, k]),
                         torch.cat([null_labels, class_labels]))
            unconditioned, conditioned = pair.chunk(2)
            epsilon = unconditioned + guidance_scale * (conditioned - unconditioned)

        alpha = alpha_bars[step]
        estimated_x0 = ((xk - (1 - alpha).sqrt() * epsilon) / alpha.sqrt()).clamp(-1, 1)
        if index + 1 == len(steps):
            xk = estimated_x0
        else:
            next_alpha = alpha_bars[steps[index + 1]]
            xk = next_alpha.sqrt() * estimated_x0 + (1 - next_alpha).sqrt() * epsilon
    return xk.clamp(-1, 1)  # [B, 3, 64, 64]


# Experiments - Modality 1 (Images)

## Experiment 1. Diffusion Baseline Evaluation

In [ ]:
# Install FID dependencies into this notebook's Python environment with uv if needed:
# uv pip install --python <path-to-this-kernel-python> torchmetrics torch-fidelity
from torchmetrics.image.fid import FrechetInceptionDistance


In [ ]:
# Keep real and generated sample counts equal for the FID comparison.
num_eval = len(val_data)  # 2,048 held-out DeepFashion validation images.
eval_batch_size = 16
eval_seed = 123
sample_steps = 50
reference_labels = [label for _, label, _ in val_data.records]


In [ ]:
# Baseline FID: generate without class guidance and compare to held-out fashion images.
def To_FID_Format(images):
    '''FID's Inception implementation expects RGB uint8 pixels in [0, 255].'''
    return ((images.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)


fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
fid.set_dtype(torch.float64)
with torch.no_grad():
    for real_images, _, captions in val_loader:
        fid.update(To_FID_Format(real_images.to(device)), real=True)

generator = torch.Generator(device=device).manual_seed(eval_seed)
if device.type == "cuda":
    torch.cuda.synchronize()
start_time = time.perf_counter()

for start in range(0, num_eval, eval_batch_size):
    count = min(eval_batch_size, num_eval - start)
    labels = torch.full((count,), active_class_ids[0], device=device, dtype=torch.long)
    generated = Sample_Images(model, labels, guidance_scale=0.0,
                              generator=generator, sample_steps=sample_steps)
    fid.update(To_FID_Format(generated), real=False)

if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start_time
print(f"Validation FID without guidance: {fid.compute().item():.2f}")
print(f"Sampling time: {elapsed:.1f} seconds for {num_eval} images")
print(f"Model evaluations per image: {sample_steps}")


## Experiment 2. Classifier-free guidance evaluation

Compare the same validation images and generation seeds with guidance scales 1 and 3.


In [ ]:
# Use the same class frequencies as the held-out reference images.
labels = torch.tensor(reference_labels, device=device, dtype=torch.long)

for guidance_scale in [1.0, 3.0]:
    guided_fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    guided_fid.set_dtype(torch.float64)
    with torch.no_grad():
        for real_images, _, captions in val_loader:
            guided_fid.update(To_FID_Format(real_images.to(device)), real=True)

    generator = torch.Generator(device=device).manual_seed(eval_seed)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    for start in range(0, num_eval, eval_batch_size):
        batch_labels = labels[start:start + eval_batch_size]
        generated = Sample_Images(model, batch_labels, guidance_scale=guidance_scale,
                                  generator=generator, sample_steps=sample_steps)
        guided_fid.update(To_FID_Format(generated), real=False)

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time
    print(f"Guidance {guidance_scale:g}: validation FID={guided_fid.compute().item():.2f}")
    print(f"Sampling time: {elapsed:.1f} seconds; model evaluations per image: "
          f"{sample_steps if guidance_scale == 1 else 2 * sample_steps}")


# Additional Experiments

In [ ]:
# Show samples from sixteen category IDs that have training images.
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

sample_labels = torch.tensor(active_class_ids[:16], device=device)
sample_generator = torch.Generator(device=device).manual_seed(eval_seed)
samples = Sample_Images(model, sample_labels, guidance_scale=3.0,
                        generator=sample_generator, sample_steps=sample_steps)
display_images = ((samples.clamp(-1, 1) + 1) / 2).cpu()
grid = make_grid(display_images, nrow=4)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.show()
